# Carga de archivos Gpkg a Athena

Este script se encarga de:

- Leer los archivos .gpkg guardados en Athena (o de forma local) resultado de la transformacion en 2_AP_ShpToGpkh.ipynb
- Preprocesado de ficheros para concordar con el formato aceptado por la tabla de Athena
- Cargar los archivos transformados a Athena

<a id='contents' />

## Tabla de contenidos

1. [Carga de librerías](#loading)
2. [Definicion de metodos](#methods)
3. [Definicion de variables](#variables)
4. [Scripts principales](#scripts)

<a id='loading' />

## Carga de librerías
[(back to top)](#contents)

In [5]:
%load_ext autoreload
%autoreload 2
%run config_notebook.py

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Directorio "src" adicionado para cargar modulos


ModuleNotFoundError: No module named 'sagemaker'

In [6]:
import awswrangler as wr
import geopandas as gpd
import boto3
import shapely
import glob
import os

ModuleNotFoundError: No module named 'geopandas'

<a id='methods' />

## Definicion de metodos
[(back to top)](#contents)

In [ ]:
def set_shapes_bucket(ssm, aws_account_env):
    env_name = "prod" if aws_account_env == "produccion" else "dev"
    if (aws_account_env != "hel001"):
        return f"nutresa-datalake-{env_name}-raw"
    else:
        return ssm.get_parameter(Name='/ml/s3/foundation/feature_store_bucket_name')['Parameter']['Value']
    
def set_monitoring_name(ssm, aws_account_env):
    return ssm.get_parameter(Name='/ml/s3/foundation/monitoring_bucket_name')['Parameter']['Value']

# Method to convert Polygon geometry data to string
def convert_to_wkt(geometry: shapely.geometry.polygon.Polygon):
    return geometry.wkt

<a id='variables' />

## Definicion de variables
[(back to top)](#contents)

In [ ]:
ssm = boto3.client('ssm', region_name='us-east-1')
aws_account_env = ssm.get_parameter(Name='/ml/misc/foundation/env_name')['Parameter']['Value']

shapes_bucket = set_shapes_bucket(ssm, aws_account_env)
print(shapes_bucket)

monitoring_bucket = set_monitoring_name(ssm, aws_account_env)
print(monitoring_bucket)

s3 = boto3.resource('s3')
bucket = s3.Bucket(monitoring_bucket)
gpkg_folder = "geoPackages"

<a id='scripts' />

## Scripts principales
[(back to top)](#contents)

Verificacion de registros en Bucket (o local)

In [ ]:
# Step 2: Read the gpkg file
gpkg_file_count = 0
gpkg_files = bucket.objects.filter(Prefix=gpkg_folder).all()
#gpkg_files = glob.iglob(f"geoPackages/*")

for file in gpkg_files:
    gpkg_file_count += 1

print(f"gpkg_file_count: {gpkg_file_count}" + "\n")

Carga de Gpkg's a Athena como CSV's

In [ ]:
%%time
# Step 3: Create an Athena table using AWS Wrangler
bucket_name = monitoring_bucket
gpkg_path = 'geoPackages/05001.gpkg'

database_name = 'db_machine_learning_helados_lab'
table_name = 'tb_ml_helados_gpkgs'
path = f's3://{bucket_name}/{gpkg_path}'

gpkg_files = glob.iglob(f"geoPackages/*")
current_file_count = 1
for file in gpkg_files:
    print("="*30 + "\n")
    print(f"gpkg_file: {file}")
    gpkg_output_path = f"tbGeoPackages/"
    gdf = gpd.read_file(file)
    
    dic_dtypes = {}
    for col in gdf.columns:
        if gdf.dtypes[col] == 'object':
            dic_dtypes[col] = "string"
        elif gdf.dtypes[col] == 'float64':
            gdf[col] = gdf[col].apply(lambda x: str(x))
            dic_dtypes[col] = "string"
        elif gdf.dtypes[col] == 'geometry':
            #print(f"geometry column type before {type(gdf[col])}")
            #print(f"geometry column 0 data type before {type(gdf[col][0])}")
            gdf[col] = gdf[col].apply(convert_to_wkt)
            dic_dtypes[col] = "string"
            #print(f"geometry column type after {type(gdf[col])}")
            #print(f"geometry column 0 data type after {type(gdf[col][0])}")

    wr.s3.to_csv(
        df=gdf,
        path=f"s3://{bucket_name}/{gpkg_output_path}",
        index=False,
        dataset=True,
        mode="append",
        database=database_name,
        table=table_name,
        dtype=dic_dtypes
    )

    
    print(f"{file} OK {current_file_count}/{gpkg_file_count}" + "\n")
    print("="*30 + "\n")
    current_file_count += 1